In [ ]:
# extract stream from osm

import osmnx as ox
import geopandas as gpd
import leafmap

In [29]:
######### collect streams for the four focused cities separately

place_name = "Dresden, Germany"
# place_name = "Jablonec nad Nisou, Czech Republic"
# place_name = "Poznań, Poland"
# place_name = "Senica, Slovakia"

streams= ox.features_from_place(place_name,tags={"waterway":"stream"})

print(streams.head())
print(f"Total streams found: {len(streams)}")

city_boundary = ox.geocode_to_gdf(place_name) 

                                                          geometry waterway  \
element id                                                                    
way     4398867  LINESTRING (13.59533 51.04207, 13.59542 51.042...   stream   
        4402184  LINESTRING (13.64715 51.03711, 13.64774 51.037...   stream   
        4417709  LINESTRING (13.68127 51.04311, 13.68139 51.043...   stream   
        4863060  LINESTRING (13.81829 50.99828, 13.81872 50.998...   stream   
        4940315  LINESTRING (13.83847 51.0675, 13.83782 51.0674...   stream   

                              name wikidata  \
element id                                    
way     4398867  Zschonergrundbach      NaN   
        4402184        Gorbitzbach      NaN   
        4417709        Gorbitzbach      NaN   
        4863060       Lockwitzbach      NaN   
        4940315    Mordgrundwasser      NaN   

                                                              note boat  \
element id                               

In [30]:
# set the center of the map
# identify the center coordinates of the city
center = city_boundary.centroid
center_lon = center.x.values[0]
center_lat = center.y.values[0]


In [52]:
# 随机选条stream
stream = streams.sample(10)
print(stream)


                                                             geometry  \
element id                                                              
way     1311391167  LINESTRING (13.85716 51.08204, 13.85732 51.082...   
        37880997    LINESTRING (13.60757 51.08438, 13.60774 51.084...   
        1188906736  LINESTRING (13.58526 51.04411, 13.58534 51.044...   
        169879917   LINESTRING (13.89961 51.10583, 13.89916 51.105...   
        35676042    LINESTRING (13.87299 51.01307, 13.87268 51.01264)   
        889197878   LINESTRING (13.88131 50.98745, 13.88184 50.98752)   
        29185738    LINESTRING (13.8066 51.15867, 13.80658 51.1589...   
        539904238   LINESTRING (13.81559 51.10325, 13.81602 51.103...   
        1311391200   LINESTRING (13.86955 51.09843, 13.86951 51.0984)   
        865913694   LINESTRING (13.84555 51.12183, 13.84555 51.121...   

                   waterway                 name wikidata note boat name:hsb  \
element id                                 

In [53]:

# visualise the stream
m = leafmap.Map(center=[center_lat, center_lon], zoom=12)
m.add_gdf(stream, layer_name="Test", style={"color": "red", "weight": 5})
m.add_gdf(streams, layer_name="All Streams", style={"color": "blue", "weight": 2})
m.add_gdf(city_boundary, layer_name="City Boundary")
m 

Map(center=[np.float64(51.06632603025163), np.float64(13.783424250027194)], controls=(ZoomControl(options=['po…

In [47]:
# 随机选条stream
stream = streams.sample(1)
print(stream)


                                                            geometry waterway  \
element id                                                                      
way     997773205  LINESTRING (13.91089 51.00207, 13.91077 51.00203)   stream   

                            name wikidata note boat name:hsb width source  \
element id                                                                  
way     997773205  Graupaer Bach      NaN  NaN  NaN      NaN   NaN    NaN   

                  layer  ... start_date tunnel:length fixme wikipedia level  \
element id               ...                                                  
way     997773205    -1  ...        NaN           NaN   NaN       NaN   NaN   

                  official_name description source:name covered  \
element id                                                        
way     997773205           NaN         NaN         NaN     NaN   

                  proposed:tunnel  
element id                         
way     9977

In [54]:
# set crs to meters
stream = stream.to_crs(epsg=3857)


In [57]:
type(stream)

geopandas.geodataframe.GeoDataFrame

In [ ]:
# Function to generate points at fixed intervals along the geometries in a GeoDataFrame
def generate_points_for_streams(gdf, distance):
    all_points = []

    for idx, row in gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue
        if geom.geom_type == 'LineString':
            # Handle a single line
            all_points.extend(_generate_points_from_line(geom, distance, idx))
        elif geom.geom_type == 'MultiLineString':
            # If there are multiple lines in one record, iterate through each
            for part in geom.geoms:
                all_points.extend(_generate_points_from_line(part, distance, idx))

    return gpd.GeoDataFrame(all_points, columns=["stream_id", "geometry"], crs=gdf.crs)

def _generate_points_from_line(line_geom, distance, stream_id):
    generated_points = []
    line_length = int(line_geom.length)
    for dist in range(0, line_length, distance):
        generated_points.append((stream_id, line_geom.interpolate(dist)))
    return generated_points

points_gdf = generate_points_for_streams(stream, 100)

print(points_gdf.head())

           stream_id                         geometry
0  (way, 1311391167)  POINT (1542571.694 6635818.694)
1  (way, 1311391167)   POINT (1542658.82 6635865.578)
2  (way, 1311391167)  POINT (1542755.182 6635878.217)
3  (way, 1311391167)  POINT (1542854.434 6635872.739)
4  (way, 1311391167)  POINT (1542953.568 6635870.359)


In [ ]:
import math
import shapely
from shapely.geometry import LineString
import geopandas as gpd

def generate_perp_lines_for_streams(gdf, distance_between_points=100, half_perp_length=20):
   
    perp_lines_data = []

    for idx, row in gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue

        # Handle single line or multiple lines in the geometry
        if geom.geom_type == 'LineString':
            segments = [geom]
        elif geom.geom_type == 'MultiLineString':
            segments = list(geom.geoms)
        else:
            continue

        for line_geom in segments:
            line_length = line_geom.length

            # Step along the line in increments of 'distance_between_points'
            dist = 0
            while dist <= line_length:
                center_point = line_geom.interpolate(dist)

                # For direction: pick a small step forward, unless we’re near the end
                step = 1.0
                if dist + step > line_length:
                    step = -1.0  # step backward if near the end

                # A second point a small step away
                near_point = line_geom.interpolate(dist + step)

                # Vector from center_point to near_point
                dx = near_point.x - center_point.x
                dy = near_point.y - center_point.y

                # Perpendicular vector (rotated 90°): e.g. (dx, dy) -> (-dy, dx)
                # Normalize that perpendicular vector
                perp_len = math.sqrt(dx*dx + dy*dy)
                if perp_len == 0:
                    dist += distance_between_points
                    continue

                # Unit perpendicular
                perp_x = -dy / perp_len
                perp_y =  dx / perp_len

                # Create endpoints by moving half_perp_length in both directions
                # from the center_point
                left_end  = shapely.geometry.Point(
                    center_point.x + perp_x * half_perp_length,
                    center_point.y + perp_y * half_perp_length
                )
                right_end = shapely.geometry.Point(
                    center_point.x - perp_x * half_perp_length,
                    center_point.y - perp_y * half_perp_length
                )

                perp_line = LineString([left_end, right_end])

                # Store in a list so we can build a GeoDataFrame
                perp_lines_data.append({
                    'stream_id': idx,
                    'geometry': perp_line
                })

                dist += distance_between_points

    # Convert the list to a GeoDataFrame
    return gpd.GeoDataFrame(perp_lines_data, crs=gdf.crs)


In [76]:
from shapely.geometry import LineString, LineString
from shapely.affinity import rotate
import math

def create_perpendicular_lines(points_gdf, line_geom, half_length=50):

    line_list = []
    
    for idx, row in points_gdf.iterrows():
        point = row.geometry
        
        # 1. Find distance along the LineString for this point
        dist_along_line = line_geom.project(point)
        
        # 2. Interpolate a small distance ahead to approximate the tangent
        small_step = 0.1  # in projected CRS units (meters, if EPSG:3857)
        ahead_dist = dist_along_line + small_step
        # Make sure we don't exceed the line length
        if ahead_dist > line_geom.length:
            ahead_dist = line_geom.length
        
        ahead_point = line_geom.interpolate(ahead_dist)
        
        # 3. Direction vector (point -> ahead_point)
        dir_x = ahead_point.x - point.x
        dir_y = ahead_point.y - point.y
        
        # 4. Rotate this direction by 90° (pi/2 radians) to get the perpendicular
        #    shapely.affinity.rotate rotates around the origin, so shift to (0,0), rotate, shift back
        #    However, a simpler approach is to just swap and negate: (x, y) -> (-y, x) or (y, -x)
        perp_dir_x = -dir_y
        perp_dir_y = dir_x
        
        # Normalize to length = 1
        length_dir = math.sqrt(perp_dir_x**2 + perp_dir_y**2)
        if length_dir == 0:
            # If the direction is zero (could happen if small_step was too small on a near-vertical line),
            # skip this point or handle specially
            continue
        
        unit_x = perp_dir_x / length_dir
        unit_y = perp_dir_y / length_dir
        
        # 5. Build the line segment from (point - half_length * direction) to (point + half_length * direction)
        start_x = point.x - half_length * unit_x
        start_y = point.y - half_length * unit_y
        end_x   = point.x + half_length * unit_x
        end_y   = point.y + half_length * unit_y
        
        segment = LineString([(start_x, start_y), (end_x, end_y)])
        line_list.append(segment)
    
    # Create a GeoDataFrame of perpendicular lines
    lines_gdf = gpd.GeoDataFrame(geometry=line_list, crs=points_gdf.crs)
    return lines_gdf

line_geometry = stream.iloc[0].geometry  # Get the actual LineString from the GeoDataFrame
perp_lines_gdf = create_perpendicular_lines(points_gdf, line_geometry, half_length=50)
perp_lines_gdf.to_file("perpendicular_lines.geojson", driver="GeoJSON")

In [77]:
print(perp_lines_gdf)



                                             geometry
0   LINESTRING (1542588.055 6635771.446, 1542555.3...
1   LINESTRING (1542674.475 6635818.093, 1542643.1...
2   LINESTRING (1542751.779 6635828.333, 1542758.5...
3   LINESTRING (1542846.386 6635823.391, 1542862.4...
4   LINESTRING (1542944.95 6635821.107, 1542962.18...
5   LINESTRING (1543073.748 6635838.399, 1543012.3...
6   LINESTRING (1543141.317 6635831.614, 1543140.8...
7   LINESTRING (1543249.223 6635842.614, 1543231.1...
8   LINESTRING (1543277.792 6635878.707, 1543376.9...
9   LINESTRING (1543449.819 6635849.203, 1543361.0...
10  LINESTRING (1514786.983 6636183.983, 1514788.4...
11  LINESTRING (1514878.303 6636224.506, 1514879.9...
12  LINESTRING (1514968.817 6636264.899, 1514970.6...
13  LINESTRING (1515047.539 6636326.421, 1515049.5...
14  LINESTRING (1515126.946 6636387.028, 1515129.1...
15  LINESTRING (1515199.098 6636454.452, 1515201.6...
16  LINESTRING (1515203.484 6636549.498, 1515206.3...
17  LINESTRING (1515202.648 

In [78]:
#visualise the points and lines


m = leafmap.Map(center=[center_lat, center_lon], zoom=12)
m.add_gdf(stream, layer_name="Stream", style={"color": "red", "weight": 5})
m.add_gdf(points_gdf, layer_name="Points", style={"color": "blue", "radius": 5})
m.add_gdf(city_boundary, layer_name="City Boundary")
m.add_gdf(perp_lines_gdf, layer_name="Perp Lines", style={"color": "green", "weight": 2})

m

Map(center=[np.float64(51.06632603025163), np.float64(13.783424250027194)], controls=(ZoomControl(options=['po…

In [79]:
# create buffer zones around the stream with 50m radius
buffered_stream = stream.copy()
buffered_stream["geometry"] = stream.buffer(50)


In [81]:
# visualise the buffer zones
m = leafmap.Map(center=[center_lat, center_lon], zoom=12)
m.add_gdf(stream, layer_name="Stream", style={"color": "red", "weight": 5})
m.add_gdf(buffered_stream, layer_name="Buffered Stream", style={"color": "blue", "weight": 2})
m.add_gdf(city_boundary, layer_name="City Boundary")
m.add_gdf(perp_lines_gdf, layer_name="Perp Lines", style={"color": "green", "weight": 2})
m

Map(center=[np.float64(51.06632603025163), np.float64(13.783424250027194)], controls=(ZoomControl(options=['po…